IMPORTACIONES Y VARIABLES QUE DEFINEN EL MODELO

In [9]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image

# TAMAÑO CLAVE PARA INCEPTIONV3
IMG_SHAPE = (299, 299) #google lo entreno con este tamaño por lo que es mas recomendable usarlo asi
BATCH_SIZE = 32 # cuantas imagenes usa de golpe para entrenar
DATASET_PATH = '../data/dataset/images'

PREPARAMOS LOS DATOS

In [2]:
train_datagen = ImageDataGenerator( # VARIABLE PARA CONVERTIR LAS IMAGENES
    preprocessing_function=preprocess_input, # ESCALA LOS PIXELES COMO INCEPTIONV3 NECESITA, SIN ESTO NO FUNCIONARIA BIEN
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    validation_split=0.2    # 20% DE LAS IMAGENES PARA VALIDACION
)

# AL TENER LAS IMAGENES DE SANAS Y ENFERMAS SEPARADAS EL FLOW_FROM_DIRECTORY NOS PERMITE CARGARLAS FACILMENTE, DETECTARA QUE QUEREMOS DIFERENCIAR DOS TIPOS DE IMAGENES QUE SERAN LAS DE NUESTRAS CARPETAS

train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SHAPE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    shuffle=True # MEZCLA LAS IMAGENES PARA QUE ENTRENE CON ENFERMAS Y SANAS
)

val_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SHAPE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    shuffle=False # NO MEZCLA LAS IMAGENES PARA QUE LA VALIDACION SEA CONSISTENTE
)

Found 1983 images belonging to 2 classes.
Found 495 images belonging to 2 classes.


CONSTRUIMOS EL MODELO

In [3]:
# CARGAMOS EL MODELO BASE INCEPTIONV3 SIN LA CAPA SUPERIOR
base_model = InceptionV3(weights='imagenet',
                         include_top=False, # Quitamos la capa clasificadora original
                         input_shape=(299, 299, 3))

# CONGELAMOS EL MODELO BASE PARA QUE NO OLVIDE LO QUE YA SABE
base_model.trainable = False

# AÑADIMOS LAS CAPAS SUPERIORES PARA NUESTRA TAREA ESPECIFICA
x = base_model.output
x = GlobalAveragePooling2D()(x) # Aplanar: convierte características 2D a vector 1D
x = Dense(1024, activation='relu')(x) # Capa densa intermedia (potente para Inception)
x = Dense(1, activation='sigmoid')(x) # Capa final: 1 neurona (0=Sana, 1=Enferma)

# UNIMOS LAS CAPAS EN EL MODELO FINAL
model = Model(inputs=base_model.input, outputs=x)

87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 8s 0us/step


COMPILAMOS Y ENTRENAMOS EL MODELO

In [4]:
#COMPILAMOS EL MODELO
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

# ENTRENAMOS EL MODELO
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50  # PONEMOS MUCHOS EPOCHS PARA QUE SE SEPA EL DATASET AL DEDILLO, LUEGO SE PUEDE AJUSTAR
)

# Guardar
model.save('discriminador_hojas_inceptionv3.h5')

Epoch 1/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 36s 541ms/step - accuracy: 0.8316 - loss: 0.4520 - val_accuracy: 0.9576 - val_loss: 0.1677
Epoch 2/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 33s 528ms/step - accuracy: 0.9309 - loss: 0.1736 - val_accuracy: 0.9475 - val_loss: 0.1504
Epoch 3/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 32s 513ms/step - accuracy: 0.9400 - loss: 0.1528 - val_accuracy: 0.9333 - val_loss: 0.1882
Epoch 4/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 31s 498ms/step - accuracy: 0.9476 - loss: 0.1257 - val_accuracy: 0.9556 - val_loss: 0.1213
Epoch 5/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 31s 501ms/step - accuracy: 0.9501 - loss: 0.1278 - val_accuracy: 0.9697 - val_loss: 0.0950
Epoch 6/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 30s 490ms/step - accuracy: 0.9415 - loss: 0.1502 - val_accuracy: 0.9455 - val_loss: 0.1535
Epoch 7/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 31s 507ms/step - accuracy: 0.9370 - loss: 0.1572 - val_accuracy: 0.9737 - val_loss: 0.0930
Epoch 8/50
62/62 ━━━━━━━━━━━━━━━━━━━━ 32s 510ms/step - accuracy: 0.9637 - loss: 0.0951 - val_accu

PREDECIMOS UNA IMAGEN REAL DE UNA HOJA

In [11]:
MODELO_PATH = 'discriminador_hojas_inceptionv3.h5'
IMAGEN_A_PROBAR = '../Data/dataset/predictionImages/imagenReal_a_predecir.jpg'
IMAGEN_A_PROBAR2 = '../Data/dataset/images/Pepper,_bell___Bacterial_spot/image (1).JPG'

# CARGAMOS EL MODELO
print("Cargando el modelo")
model = tf.keras.models.load_model(MODELO_PATH)
print("¡Modelo cargado exitosamente!")

def cargar_y_preparar(ruta_imagen):
    img = image.load_img(ruta_imagen, target_size=(299, 299))
    img_array = image.img_to_array(img)

    img_array = np.expand_dims(img_array, axis=0)

    img_preprocesada = preprocess_input(img_array)

    return img_preprocesada

Cargando el modelo


¡Modelo cargado exitosamente!


In [13]:
img_lista = cargar_y_preparar(IMAGEN_A_PROBAR)
prediccion = model.predict(img_lista)

resultado_numerico = prediccion[0][0]

print(f"\n--- RESULTADO ---")
print(f"Valor numérico crudo: {resultado_numerico:.4f}")

# COMO EL NOMBRE DE LA CARPETA DE LAS ENFERMAS VA ANTES ALFABAETICAMENTE QUE LA DE LAS SANAS, INCEPTIONV3 DEVUELVE VALORES CERCANOS A 0 PARA ENFERMAS Y CERCANOS A 1 PARA SANAS.

# Usamos 0.5 como punto de corte.
if resultado_numerico < 0.5:
    confianza = (1 - resultado_numerico) * 100
    print(f"Diagnóstico: 🍂 ENFERMA")
    print(f"Seguridad: {confianza:.2f}%")
else:
    confianza = resultado_numerico * 100
    print(f"Diagnóstico: 🌿 SANA")
    print(f"Seguridad: {confianza:.2f}%")

# con IMAGEN_A_PROBAR2
#--- RESULTADO ---
#Valor numérico crudo: 0.0000
#Diagnóstico: 🍂 ENFERMA
#Seguridad: 100.00%
#
# CON IMAGEN_A_PROBAR
#--- RESULTADO ---
#Valor numérico crudo: 0.0669
#Diagnóstico: 🍂 ENFERMA
#Seguridad: 93.31%


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step

--- RESULTADO ---
Valor numérico crudo: 0.0669
Diagnóstico: 🍂 ENFERMA
Seguridad: 93.31%
